# 프로젝트 3 - Weekend 2: 정답지 (실행 가능 버전)

> 각 문제 셀(`# 문제 N`) 바로 뒤에 정답 셀(`# ✅ 문제 N 정답`)이 따라옵니다. 처음부터 끝까지 Run All 하면 모든 문제가 풀린 상태로 진행됩니다.

**프로젝트**: 법률 문서 기반 검색 에이전트 시스템 (Weekend 2/3)

**학습 목표**:
1. `StateGraph` + `TypedDict`로 법률 에이전트 상태 시스템 설계
2. 노드·엣지를 분리하여 `classify → search → analyze → answer` 그래프 구축
3. 조건부 라우팅과 ReAct 패턴으로 질문 유형별 최적 경로 구현
4. `MemorySaver`로 대화 이력 영속화 + Human-in-the-Loop

In [ ]:
# 환경 설정
!pip install -q langchain langchain-openai langchain-community faiss-cpu \
    langgraph pandas numpy python-dotenv

In [ ]:
import os
import json
import time
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage, BaseMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ 환경 설정 완료")

---
## 문제 1: Weekend 1 자산 복원

Weekend 2는 Weekend 1의 벡터스토어와 도구를 재사용합니다. 아래 셀에 **간소화된 복원 코드**를 작성하세요.

**요구사항:**
- `SAMPLE_LAW_ARTICLES` 리스트 (Weekend 1과 동일 데이터, 5개 이상 조문 포함)
- `law_docs`: `Document` 리스트 구축
- `vector_store`: FAISS 벡터스토어 구축
- `@tool` `search_law(query, top_k=3)`: 유사도 검색 도구
- 검증 함수 `verify_weekend1_assets()` 호출 → 체크포인트 출력

**평가기준:**
- `vector_store.similarity_search("이혼")` 결과 존재
- `search_law.name == "search_law"`

In [ ]:
# 문제 1: Weekend 1 자산 복원 (간소화 버전)

# ---- 여기에 코드 작성 ----
# 1) SAMPLE_LAW_ARTICLES 정의 (아래 샘플 사용 가능)
SAMPLE_LAW_ARTICLES = [
    {"law": "민법", "article": "제750조", "title": "불법행위의 내용",
     "content": "고의 또는 과실로 인한 위법행위로 타인에게 손해를 가한 자는 그 손해를 배상할 책임이 있다."},
    {"law": "민법", "article": "제840조", "title": "재판상 이혼원인",
     "content": "부부의 일방은 1.부정한 행위 2.악의의 유기 3.부당한 대우 등의 사유가 있는 경우 이혼을 청구할 수 있다."},
    {"law": "형법", "article": "제307조", "title": "명예훼손",
     "content": "공연히 사실을 적시하여 사람의 명예를 훼손한 자는 2년 이하의 징역이나 500만원 이하의 벌금에 처한다."},
    {"law": "형법", "article": "제329조", "title": "절도",
     "content": "타인의 재물을 절취한 자는 6년 이하의 징역 또는 1천만원 이하의 벌금에 처한다."},
    {"law": "형법", "article": "제347조", "title": "사기",
     "content": "사람을 기망하여 재물의 교부를 받거나 재산상의 이익을 취득한 자는 10년 이하의 징역 또는 2천만원 이하의 벌금에 처한다."},
    {"law": "상법", "article": "제382조", "title": "이사의 선임, 임기",
     "content": "이사는 주주총회에서 선임하며, 임기는 3년을 초과하지 못한다."},
]

# 2) law_docs 구축
law_docs = None  # Document 리스트로 변환

# 3) vector_store 구축
vector_store = None  # FAISS.from_documents() 사용

# 4) @tool search_law 정의
# @tool
# def search_law(query: str, top_k: int = 3) -> str:
#     ...

def verify_weekend1_assets():
    """Weekend 1 자산 복원 검증."""
    checks = {
        "SAMPLE_LAW_ARTICLES": len(SAMPLE_LAW_ARTICLES) >= 5,
        "law_docs": law_docs is not None and len(law_docs) >= 5,
        "vector_store": vector_store is not None,
        "search_law tool": 'search_law' in dir() and hasattr(search_law, 'name'),
    }
    for name, ok in checks.items():
        print(f"  {'✅' if ok else '❌'} {name}")
    return all(checks.values())

verify_weekend1_assets()

In [ ]:
# ✅ 문제 1 정답
SAMPLE_LAW_ARTICLES = [
    {"law": "민법", "article": "제750조", "title": "불법행위의 내용",
     "content": "고의 또는 과실로 인한 위법행위로 타인에게 손해를 가한 자는 그 손해를 배상할 책임이 있다."},
    {"law": "민법", "article": "제840조", "title": "재판상 이혼원인",
     "content": "부부의 일방은 1.부정한 행위 2.악의의 유기 3.부당한 대우 등의 사유가 있는 경우 이혼을 청구할 수 있다."},
    {"law": "형법", "article": "제307조", "title": "명예훼손",
     "content": "공연히 사실을 적시하여 사람의 명예를 훼손한 자는 2년 이하의 징역이나 500만원 이하의 벌금에 처한다."},
    {"law": "형법", "article": "제329조", "title": "절도",
     "content": "타인의 재물을 절취한 자는 6년 이하의 징역 또는 1천만원 이하의 벌금에 처한다."},
    {"law": "형법", "article": "제347조", "title": "사기",
     "content": "사람을 기망하여 재물의 교부를 받거나 재산상의 이익을 취득한 자는 10년 이하의 징역 또는 2천만원 이하의 벌금에 처한다."},
    {"law": "상법", "article": "제382조", "title": "이사의 선임, 임기",
     "content": "이사는 주주총회에서 선임하며, 임기는 3년을 초과하지 못한다."},
]

law_docs = [
    Document(
        page_content=f"{a['law']} {a['article']} ({a['title']}): {a['content']}",
        metadata={"law": a["law"], "article": a["article"], "title": a["title"]},
    )
    for a in SAMPLE_LAW_ARTICLES
]

vector_store = FAISS.from_documents(law_docs, embeddings)

@tool
def search_law(query: str, top_k: int = 3) -> str:
    """법률 조문을 의미 기반으로 검색합니다."""
    results = vector_store.similarity_search(query, k=top_k)
    out = [{"law": r.metadata["law"], "article": r.metadata["article"],
            "title": r.metadata["title"], "snippet": r.page_content[:120]}
           for r in results]
    return json.dumps(out, ensure_ascii=False, indent=2)

def verify_weekend1_assets():
    checks = {
        "SAMPLE_LAW_ARTICLES": len(SAMPLE_LAW_ARTICLES) >= 5,
        "law_docs": law_docs is not None and len(law_docs) >= 5,
        "vector_store": vector_store is not None,
        "search_law tool": hasattr(search_law, 'name') and search_law.name == "search_law",
    }
    for name, ok in checks.items():
        print(f"  {'✅' if ok else '❌'} {name}")
    return all(checks.values())

verify_weekend1_assets()

---
## 문제 2: `LegalAgentState` 정의

LangGraph가 노드 간에 전달할 상태를 `TypedDict`로 정의하세요.

**요구사항:**
- `LegalAgentState`라는 TypedDict 클래스
- 필드:
  - `messages`: `Annotated[list[BaseMessage], add_messages]` (LangGraph가 자동 병합)
  - `query`: `str` (원본 질문)
  - `question_type`: `Literal["article_search", "case_search", "term_explain", "general"]`
  - `search_results`: `list[dict]` (검색된 조문)
  - `answer`: `str` (최종 답변)

**평가기준:**
- `LegalAgentState`가 `TypedDict`의 서브클래스
- 5개 필드가 모두 정의됨
- `Annotated[..., add_messages]`가 messages에 적용됨

In [ ]:
# 문제 2: LegalAgentState
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class LegalAgentState(TypedDict):
    # ---- 여기에 코드 작성 ----
    pass

# 테스트: 샘플 상태 생성
sample_state = LegalAgentState(
    messages=[HumanMessage(content="이혼 사유는?")],
    query="이혼 사유는?",
    question_type="article_search",
    search_results=[],
    answer="",
)
print("✅ LegalAgentState 필드:", list(LegalAgentState.__annotations__.keys()))

In [ ]:
# ✅ 문제 2 정답
class LegalAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    query: str
    question_type: Literal["article_search", "case_search", "term_explain", "general"]
    search_results: list
    answer: str

# 확인
print("✅ 필드:", list(LegalAgentState.__annotations__.keys()))

---
## 문제 3: `classify_node` — 질문 유형 분류 노드

질문을 4개 유형 중 하나로 분류하는 노드 함수를 구현하세요.

**요구사항:**
- 함수 시그니처: `classify_node(state: LegalAgentState) -> dict`
- LLM에게 질문 유형 분류 요청 (프롬프트에 4개 타입 후보 명시)
- 반환값: `{"question_type": "article_search" | "case_search" | "term_explain" | "general"}`
- LLM이 예상 밖 응답 반환 시 "general"로 fallback

**평가기준:**
- "이혼 사유는?" → "article_search"
- "이혼 관련 판례 알려줘" → "case_search"
- "미필적 고의가 뭐야?" → "term_explain"
- "안녕" → "general"

In [ ]:
# 문제 3: classify_node
def classify_node(state):
    """질문 유형을 분류합니다."""
    query = state["query"]

    # ---- 여기에 코드 작성 ----
    # 1) LLM에게 분류 요청 프롬프트 작성
    # 2) response.content.strip().lower() 등으로 결과 정규화
    # 3) 4개 유형 중 하나가 아니면 "general"로 fallback
    question_type = "general"

    return {"question_type": question_type}

# 테스트
test_queries = [
    "이혼 사유는 뭐가 있어?",
    "명예훼손 관련 판례 알려줘",
    "미필적 고의가 뭐야?",
    "안녕하세요",
]
for q in test_queries:
    state = LegalAgentState(messages=[], query=q, question_type="", search_results=[], answer="")
    result = classify_node(state)
    print(f"  '{q}' → {result['question_type']}")

In [ ]:
# ✅ 문제 3 정답
VALID_TYPES = {"article_search", "case_search", "term_explain", "general"}

def classify_node(state):
    query = state["query"]
    prompt = f"""다음 질문을 아래 4개 중 하나로 분류하세요. 답변은 영문 소문자 한 단어만.

article_search: 법률 조문 검색이 필요한 질문 (예: "이혼 사유는?", "명예훼손 처벌")
case_search: 판례 검색이 필요한 질문 (예: "~ 판례 알려줘", "~ 관련 사례")
term_explain: 법률 용어 정의를 묻는 질문 (예: "미필적 고의가 뭐야?")
general: 법률과 무관하거나 인사 (예: "안녕하세요")

질문: {query}
분류:"""
    response = llm.invoke([HumanMessage(content=prompt)])
    label = response.content.strip().lower().split()[0].strip(".,")
    if label not in VALID_TYPES:
        label = "general"
    return {"question_type": label}

# 테스트
for q in ["이혼 사유는 뭐가 있어?", "명예훼손 관련 판례 알려줘", "미필적 고의가 뭐야?", "안녕하세요"]:
    state = LegalAgentState(messages=[], query=q, question_type="general", search_results=[], answer="")
    print(f"  '{q}' → {classify_node(state)['question_type']}")

---
## 문제 4: `search_node` + `analyze_node`

검색 노드와 분석 노드를 각각 구현하세요.

**요구사항:**

**`search_node(state)`**:
- `state["query"]`로 vector_store 검색 (k=3)
- 결과를 `search_results` 필드에 dict 리스트로 저장: `[{"law":..., "article":..., "title":..., "content":...}]`

**`analyze_node(state)`**:
- `state["search_results"]`를 LLM 프롬프트에 포함
- 질문에 답변 생성
- 반환: `{"messages": [AIMessage(content=answer)], "answer": answer}`

**평가기준:**
- `search_node` 반환값에 `search_results` 키 존재 (리스트 3개)
- `analyze_node` 반환값에 `messages`와 `answer` 키 존재

In [ ]:
# 문제 4: search_node + analyze_node
def search_node(state):
    """벡터 검색으로 관련 조문을 찾습니다."""
    query = state["query"]
    # ---- 여기에 코드 작성 ----
    search_results = []
    return {"search_results": search_results}

def analyze_node(state):
    """검색 결과를 바탕으로 답변을 생성합니다."""
    query = state["query"]
    results = state["search_results"]
    # ---- 여기에 코드 작성 ----
    # 1) 검색 결과를 프롬프트에 포맷팅
    # 2) LLM에 답변 요청
    # 3) AIMessage와 answer 반환
    answer = "미구현"
    return {"messages": [AIMessage(content=answer)], "answer": answer}

# 테스트 (search만)
test_state = LegalAgentState(
    messages=[], query="이혼 사유", question_type="article_search",
    search_results=[], answer=""
)
search_out = search_node(test_state)
print(f"검색 결과 {len(search_out.get('search_results', []))}개")
for r in search_out.get('search_results', [])[:2]:
    print(f"  - {r.get('law')} {r.get('article')}")

In [ ]:
# ✅ 문제 4 정답
def search_node(state):
    query = state["query"]
    docs = vector_store.similarity_search(query, k=3)
    results = [{
        "law": d.metadata.get("law"),
        "article": d.metadata.get("article"),
        "title": d.metadata.get("title"),
        "content": d.page_content,
    } for d in docs]
    return {"search_results": results}

def analyze_node(state):
    query = state["query"]
    results = state["search_results"]
    if not results:
        answer = "관련 조문을 찾지 못했습니다."
    else:
        context = "\n\n".join([f"- {r['law']} {r['article']}: {r['content']}" for r in results])
        prompt = f"""다음 법률 조문을 참고하여 질문에 답하세요.

질문: {query}

참고 조문:
{context}

답변 (3-5문장):"""
        response = llm.invoke([HumanMessage(content=prompt)])
        answer = response.content
    return {"messages": [AIMessage(content=answer)], "answer": answer}

# 테스트
state = LegalAgentState(messages=[], query="이혼 사유", question_type="article_search",
                        search_results=[], answer="")
sr = search_node(state)
print(f"검색 결과 {len(sr['search_results'])}개")
for r in sr["search_results"][:2]:
    print(f"  - {r['law']} {r['article']}")

---
## 문제 5: StateGraph 기본 구축

`classify → search → analyze` 순서로 노드를 연결한 그래프를 만드세요.

**요구사항:**
- `StateGraph(LegalAgentState)`로 그래프 빌더 생성
- 3개 노드 등록 (`classify`, `search`, `analyze`)
- 엣지 연결: `START → classify → search → analyze → END`
- `.compile()`로 실행 가능한 그래프 반환
- 함수 시그니처: `build_basic_graph() -> CompiledStateGraph`

**평가기준:**
- `graph.invoke({...})` 호출 가능
- 최종 state에 `answer` 필드가 채워짐

In [ ]:
# 문제 5: StateGraph 기본 구축
def build_basic_graph():
    """classify → search → analyze 순서 그래프."""
    builder = StateGraph(LegalAgentState)
    # ---- 여기에 코드 작성 ----
    # 1) 노드 3개 추가 (add_node)
    # 2) 엣지 4개 연결 (START → classify → search → analyze → END)
    # 3) builder.compile() 반환
    return None

# 테스트
graph = build_basic_graph()
if graph:
    initial = {
        "messages": [HumanMessage(content="명예훼손 처벌은?")],
        "query": "명예훼손 처벌은?",
        "question_type": "",
        "search_results": [],
        "answer": "",
    }
    result = graph.invoke(initial)
    print(f"🤖 질문 유형: {result['question_type']}")
    print(f"📚 검색 결과: {len(result['search_results'])}개")
    print(f"💬 답변: {result['answer'][:200]}")

In [ ]:
# ✅ 문제 5 정답
def build_basic_graph():
    builder = StateGraph(LegalAgentState)
    builder.add_node("classify", classify_node)
    builder.add_node("search", search_node)
    builder.add_node("analyze", analyze_node)
    builder.add_edge(START, "classify")
    builder.add_edge("classify", "search")
    builder.add_edge("search", "analyze")
    builder.add_edge("analyze", END)
    return builder.compile()

# 테스트
graph = build_basic_graph()
result = graph.invoke({
    "messages": [HumanMessage(content="명예훼손 처벌은?")],
    "query": "명예훼손 처벌은?",
    "question_type": "", "search_results": [], "answer": "",
})
print(f"🤖 유형: {result['question_type']}, 검색: {len(result['search_results'])}개")
print(f"💬 {result['answer'][:200]}")

---
## 문제 6: 조건부 라우팅 — 질문 유형별 분기

질문 유형에 따라 다른 노드로 분기하도록 그래프를 확장하세요.

**요구사항:**
- `term_node(state)`: 용어 설명 전용 노드 (간단한 LLM 호출)
- `general_node(state)`: 일반 질문 대답 노드
- 라우터 함수 `route_by_type(state)`: state["question_type"]에 따라 다음 노드명 반환
  - `article_search` / `case_search` → `"search"`
  - `term_explain` → `"term"`
  - `general` → `"general"`
- `add_conditional_edges(classify, route_by_type, {...})` 로 분기
- 함수 시그니처: `build_routed_graph() -> CompiledStateGraph`

**평가기준:**
- "이혼" → search 경로
- "미필적 고의" → term 경로
- "안녕" → general 경로

In [ ]:
# 문제 6: 조건부 라우팅 그래프
def term_node(state):
    """용어 설명 전용 노드."""
    # ---- 여기에 코드 작성 ----
    answer = "미구현"
    return {"messages": [AIMessage(content=answer)], "answer": answer}

def general_node(state):
    """일반 질문 노드."""
    # ---- 여기에 코드 작성 ----
    answer = "미구현"
    return {"messages": [AIMessage(content=answer)], "answer": answer}

def route_by_type(state):
    """질문 유형으로 다음 노드를 결정."""
    # ---- 여기에 코드 작성 ----
    return "general"

def build_routed_graph():
    builder = StateGraph(LegalAgentState)
    # ---- 여기에 코드 작성 ----
    # 1) 노드 추가: classify, search, analyze, term, general
    # 2) START → classify
    # 3) classify에서 add_conditional_edges(route_by_type, {...})
    # 4) search → analyze → END
    # 5) term → END, general → END
    return None

# 테스트
graph = build_routed_graph()
if graph:
    for q in ["이혼 사유는?", "미필적 고의가 뭐야?", "안녕하세요"]:
        initial = {
            "messages": [HumanMessage(content=q)], "query": q,
            "question_type": "", "search_results": [], "answer": "",
        }
        result = graph.invoke(initial)
        print(f"  '{q}' [{result['question_type']}] → {result['answer'][:80]}...")

In [ ]:
# ✅ 문제 6 정답
def term_node(state):
    prompt = f"법률 용어 '{state['query']}'를 2-3문장으로 쉽게 설명하세요."
    answer = llm.invoke([HumanMessage(content=prompt)]).content
    return {"messages": [AIMessage(content=answer)], "answer": answer}

def general_node(state):
    prompt = f"다음 질문에 간단히 답하세요: {state['query']}"
    answer = llm.invoke([HumanMessage(content=prompt)]).content
    return {"messages": [AIMessage(content=answer)], "answer": answer}

def route_by_type(state):
    t = state["question_type"]
    if t in ("article_search", "case_search"):
        return "search"
    if t == "term_explain":
        return "term"
    return "general"

def build_routed_graph():
    builder = StateGraph(LegalAgentState)
    builder.add_node("classify", classify_node)
    builder.add_node("search", search_node)
    builder.add_node("analyze", analyze_node)
    builder.add_node("term", term_node)
    builder.add_node("general", general_node)

    builder.add_edge(START, "classify")
    builder.add_conditional_edges(
        "classify", route_by_type,
        {"search": "search", "term": "term", "general": "general"}
    )
    builder.add_edge("search", "analyze")
    builder.add_edge("analyze", END)
    builder.add_edge("term", END)
    builder.add_edge("general", END)
    return builder.compile()

graph = build_routed_graph()
for q in ["이혼 사유는?", "미필적 고의가 뭐야?", "안녕하세요"]:
    r = graph.invoke({
        "messages": [HumanMessage(content=q)], "query": q,
        "question_type": "", "search_results": [], "answer": "",
    })
    print(f"  '{q}' [{r['question_type']}] → {r['answer'][:80]}...")

---
## 문제 7: ReAct 패턴 — 도구 호출 루프

LLM이 필요에 따라 `search_law` 도구를 반복 호출하는 ReAct 패턴 그래프를 구현하세요.

**요구사항:**
- `agent_node(state)`: `llm_with_tools.invoke(messages)` 호출 후 메시지 반환
- `tool_node(state)`: 마지막 AIMessage의 tool_calls를 실행하여 ToolMessage 반환
- `should_continue(state)`: 마지막 메시지에 tool_calls 있으면 `"tools"`, 없으면 `END`
- 그래프: `START → agent ⇄ tools`, agent의 조건부 엣지로 종료 결정
- 함수 시그니처: `build_react_graph(tools: list) -> CompiledStateGraph`

**평가기준:**
- 도구 호출이 필요한 질문 시 tool_node 실행 기록
- 최종 응답에 AIMessage 포함

In [ ]:
# 문제 7: ReAct 패턴
def agent_node(state):
    """LLM이 응답 또는 도구 호출을 결정."""
    # ---- 여기에 코드 작성 ----
    # 1) tools를 bind_tools로 바인딩한 llm_with_tools 사용 (클로저 or 전역 변수)
    # 2) response = llm_with_tools.invoke(state["messages"])
    # 3) {"messages": [response]} 반환 (add_messages가 자동 append)
    return {"messages": []}

def tool_node(state):
    """마지막 AIMessage의 tool_calls 실행."""
    # ---- 여기에 코드 작성 ----
    # 1) last_msg = state["messages"][-1]
    # 2) tool_calls 순회 → tool_map으로 실행 → ToolMessage 생성
    return {"messages": []}

def should_continue(state):
    """도구 호출 여부로 계속 실행 or 종료."""
    # ---- 여기에 코드 작성 ----
    return END

def build_react_graph(tools):
    tool_map = {t.name: t for t in tools}
    llm_with_tools = llm.bind_tools(tools)

    # 전역으로 저장 (노드 함수에서 접근)
    global _react_tool_map, _react_llm_with_tools
    _react_tool_map = tool_map
    _react_llm_with_tools = llm_with_tools

    builder = StateGraph(LegalAgentState)
    # ---- 여기에 코드 작성 ----
    # 1) agent 노드, tools 노드 추가
    # 2) START → agent
    # 3) agent에서 should_continue로 조건부 엣지: "tools" → tools, END → END
    # 4) tools → agent
    return None

# 테스트 (search_law만 준비되어 있다면)
if 'search_law' in dir():
    graph = build_react_graph([search_law])
    if graph:
        initial = {
            "messages": [HumanMessage(content="사기죄 형량을 알려줘")],
            "query": "사기죄 형량을 알려줘",
            "question_type": "", "search_results": [], "answer": "",
        }
        result = graph.invoke(initial)
        print(f"최종 메시지 수: {len(result['messages'])}")
        print(f"마지막 답변: {result['messages'][-1].content[:200]}")

In [ ]:
# ✅ 문제 7 정답
_react_tool_map = {}
_react_llm_with_tools = None

def agent_node(state):
    response = _react_llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def tool_node(state):
    last = state["messages"][-1]
    new_messages = []
    for tc in last.tool_calls:
        tool_fn = _react_tool_map.get(tc["name"])
        try:
            result = tool_fn.invoke(tc["args"]) if tool_fn else f"도구 '{tc['name']}' 없음"
        except Exception as e:
            result = f"도구 오류: {e}"
        new_messages.append(ToolMessage(content=str(result)[:3000], tool_call_id=tc["id"]))
    return {"messages": new_messages}

def should_continue(state):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

def build_react_graph(tools):
    global _react_tool_map, _react_llm_with_tools
    _react_tool_map = {t.name: t for t in tools}
    _react_llm_with_tools = llm.bind_tools(tools)

    builder = StateGraph(LegalAgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", tool_node)
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile()

graph = build_react_graph([search_law])
result = graph.invoke({
    "messages": [HumanMessage(content="사기죄 형량을 알려줘")],
    "query": "사기죄 형량을 알려줘",
    "question_type": "", "search_results": [], "answer": "",
})
print(f"최종 메시지 수: {len(result['messages'])}")
print(f"🤖 {result['messages'][-1].content[:200]}")

---
## 문제 8: `MemorySaver` — 대화 이력 영속화

`MemorySaver`를 체크포인터로 사용하여 `thread_id` 별로 대화 이력을 유지하세요.

**요구사항:**
- `build_memory_graph(tools)`: ReAct 그래프에 `checkpointer=MemorySaver()` 적용
- 테스트: 같은 `thread_id`로 2번 호출 시 이전 대화 참고 가능
- `config={"configurable": {"thread_id": "user1"}}` 형식으로 실행
- `graph.get_state(config)`로 현재 상태 조회 가능 확인

**평가기준:**
- 첫 질문 후 두 번째 질문에서 맥락 유지
- `graph.get_state(config).values["messages"]` 길이가 누적됨

In [ ]:
# 문제 8: MemorySaver 통합
def build_memory_graph(tools):
    """MemorySaver를 체크포인터로 사용하는 그래프."""
    # ---- 여기에 코드 작성 ----
    # 1) build_react_graph와 유사하게 빌더 구성
    # 2) builder.compile(checkpointer=MemorySaver()) 로 컴파일
    return None

# 테스트
if 'search_law' in dir():
    graph = build_memory_graph([search_law])
    if graph:
        config = {"configurable": {"thread_id": "session-demo"}}

        # 첫 질문
        print("👤 Q1: 명예훼손 처벌은?")
        r1 = graph.invoke({
            "messages": [HumanMessage(content="명예훼손 처벌은?")],
            "query": "명예훼손 처벌은?",
            "question_type": "", "search_results": [], "answer": "",
        }, config=config)
        print(f"🤖:", r1["messages"][-1].content[:150])

        # 같은 세션 2번째 질문
        print("\n👤 Q2: 허위 사실이면 어떻게 달라져?")
        r2 = graph.invoke({
            "messages": [HumanMessage(content="허위 사실이면 어떻게 달라져?")],
            "query": "허위 사실이면 어떻게 달라져?",
            "question_type": "", "search_results": [], "answer": "",
        }, config=config)
        print(f"🤖:", r2["messages"][-1].content[:150])

        # 상태 조회
        state = graph.get_state(config)
        print(f"\n📝 누적 메시지 수: {len(state.values['messages'])}")

In [ ]:
# ✅ 문제 8 정답
def build_memory_graph(tools):
    global _react_tool_map, _react_llm_with_tools
    _react_tool_map = {t.name: t for t in tools}
    _react_llm_with_tools = llm.bind_tools(tools)

    builder = StateGraph(LegalAgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", tool_node)
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile(checkpointer=MemorySaver())

graph = build_memory_graph([search_law])
config = {"configurable": {"thread_id": "session-demo"}}

print("👤 Q1: 명예훼손 처벌은?")
r1 = graph.invoke({
    "messages": [HumanMessage(content="명예훼손 처벌은?")],
    "query": "명예훼손 처벌은?",
    "question_type": "", "search_results": [], "answer": "",
}, config=config)
print(f"🤖:", r1["messages"][-1].content[:150])

print("\n👤 Q2: 허위 사실이면 어떻게 달라져?")
r2 = graph.invoke({
    "messages": [HumanMessage(content="허위 사실이면 어떻게 달라져?")],
    "query": "허위 사실이면 어떻게 달라져?",
    "question_type": "", "search_results": [], "answer": "",
}, config=config)
print(f"🤖:", r2["messages"][-1].content[:150])

state = graph.get_state(config)
print(f"\n📝 누적 메시지 수: {len(state.values['messages'])}")

---
## 문제 9: Human-in-the-Loop — 도구 실행 전 사용자 확인

민감한 도구 실행 전에 사용자 승인을 받는 패턴을 구현하세요.

**요구사항:**
- `build_hitl_graph(tools)`: `interrupt_before=["tools"]` 옵션으로 컴파일
- 실행 시 tools 노드 앞에서 자동 정지
- `graph.get_state(config)` 로 현재 상태 확인
- `graph.invoke(None, config=config)`로 재개

**평가기준:**
- 첫 `invoke()`는 도구 실행 전에 멈춤 (answer가 비어있음)
- 두 번째 `invoke(None, config)`로 재개하면 답변 완성

In [ ]:
# 문제 9: Human-in-the-Loop
def build_hitl_graph(tools):
    """도구 실행 전 interrupt하는 그래프."""
    # ---- 여기에 코드 작성 ----
    # build_memory_graph와 유사하지만
    # builder.compile(checkpointer=..., interrupt_before=["tools"])
    return None

# 테스트
if 'search_law' in dir():
    graph = build_hitl_graph([search_law])
    if graph:
        config = {"configurable": {"thread_id": "hitl-demo"}}

        # 1단계: 도구 호출 직전까지만 실행
        print("=== 1단계: 도구 실행 전 정지 ===")
        r1 = graph.invoke({
            "messages": [HumanMessage(content="사기죄 형량 알려줘")],
            "query": "사기죄 형량 알려줘",
            "question_type": "", "search_results": [], "answer": "",
        }, config=config)

        state = graph.get_state(config)
        print(f"다음 실행 노드: {state.next}")
        print(f"대기 중인 tool_calls: {state.values['messages'][-1].tool_calls if state.values['messages'] else 'N/A'}")

        # 2단계: 사용자 승인 후 재개
        print("\n=== 2단계: 재개 (도구 실행) ===")
        r2 = graph.invoke(None, config=config)
        print(f"🤖 최종: {r2['messages'][-1].content[:200]}")

In [ ]:
# ✅ 문제 9 정답
def build_hitl_graph(tools):
    global _react_tool_map, _react_llm_with_tools
    _react_tool_map = {t.name: t for t in tools}
    _react_llm_with_tools = llm.bind_tools(tools)

    builder = StateGraph(LegalAgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", tool_node)
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile(
        checkpointer=MemorySaver(),
        interrupt_before=["tools"],
    )

graph = build_hitl_graph([search_law])
config = {"configurable": {"thread_id": "hitl-demo"}}

print("=== 1단계: 도구 실행 전 정지 ===")
r1 = graph.invoke({
    "messages": [HumanMessage(content="사기죄 형량 알려줘")],
    "query": "사기죄 형량 알려줘",
    "question_type": "", "search_results": [], "answer": "",
}, config=config)
state = graph.get_state(config)
print(f"다음 실행 노드: {state.next}")

print("\n=== 2단계: 재개 ===")
r2 = graph.invoke(None, config=config)
print(f"🤖 최종: {r2['messages'][-1].content[:200]}")

---
## 문제 10: 미니 프로젝트 — `LegalGraphAgent` 통합

Weekend 2의 모든 요소를 통합한 `LegalGraphAgent` 클래스를 완성하세요.

**요구사항:**
- 내부에 MemorySaver 기반 ReAct 그래프 + 조건부 라우팅 포함
- 메서드:
  - `ask(query, thread_id="default") -> dict`: 답변 + 상태 정보 반환
  - `visualize() -> str`: `graph.get_graph().draw_mermaid()` 로 그래프 구조 출력
  - `get_history(thread_id) -> list`: 해당 세션의 대화 이력 반환
- 최소 2개 thread로 멀티 세션 시연

**평가기준:**
- 서로 다른 thread_id는 이력 분리
- `visualize()` 결과에 노드/엣지 구조 표시
- 최종 답변에 면책 조항 (간단한 문구) 포함


In [ ]:
# 문제 10: LegalGraphAgent
class LegalGraphAgent:
    """Weekend 2 최종 결과물 — LangGraph 기반 법률 에이전트."""

    def __init__(self, tools):
        self.tools = tools
        self.tool_map = {t.name: t for t in tools}
        self.llm_with_tools = llm.bind_tools(tools)
        # ---- 여기에 코드 작성 ----
        # 1) self.graph = self._build_graph()
        pass

    def _build_graph(self):
        """ReAct + MemorySaver 그래프 구축."""
        # ---- 여기에 코드 작성 ----
        return None

    def ask(self, query, thread_id="default"):
        """질문에 답하며 세션별 이력 유지."""
        start = time.time()
        config = {"configurable": {"thread_id": thread_id}}
        # ---- 여기에 코드 작성 ----
        # 1) graph.invoke로 실행
        # 2) 답변에 "⚠️ 법률 자문 아님" 문구 추가
        # 3) {"answer": ..., "thread_id": ..., "elapsed": ..., "messages_count": ...} 반환
        return {"answer": "미구현", "thread_id": thread_id, "elapsed": 0, "messages_count": 0}

    def visualize(self):
        """그래프 구조를 Mermaid로 출력."""
        # ---- 여기에 코드 작성 ----
        return ""

    def get_history(self, thread_id="default"):
        """해당 세션의 메시지 이력."""
        # ---- 여기에 코드 작성 ----
        return []

# 테스트
if 'search_law' in dir():
    agent = LegalGraphAgent([search_law])
    print("=== 그래프 구조 ===")
    print(agent.visualize())

    print("\n=== Thread A ===")
    print(agent.ask("명예훼손 처벌은?", thread_id="user-A")["answer"][:150])

    print("\n=== Thread B (다른 세션) ===")
    print(agent.ask("사기죄는?", thread_id="user-B")["answer"][:150])

    print("\n=== Thread A 이력 길이:", len(agent.get_history("user-A")))
    print("=== Thread B 이력 길이:", len(agent.get_history("user-B")))

In [ ]:
# ✅ 문제 10 정답
class LegalGraphAgent:
    """Weekend 2 최종 결과물."""

    DISCLAIMER = "\n\n⚠️ 본 답변은 일반 정보이며 법률 자문이 아닙니다."

    def __init__(self, tools):
        self.tools = tools
        self.tool_map = {t.name: t for t in tools}
        self.llm_with_tools = llm.bind_tools(tools)
        self.memory = MemorySaver()
        self.graph = self._build_graph()

    def _build_graph(self):
        global _react_tool_map, _react_llm_with_tools
        _react_tool_map = self.tool_map
        _react_llm_with_tools = self.llm_with_tools

        builder = StateGraph(LegalAgentState)
        builder.add_node("agent", agent_node)
        builder.add_node("tools", tool_node)
        builder.add_edge(START, "agent")
        builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
        builder.add_edge("tools", "agent")
        return builder.compile(checkpointer=self.memory)

    def ask(self, query, thread_id="default"):
        start = time.time()
        config = {"configurable": {"thread_id": thread_id}}
        result = self.graph.invoke({
            "messages": [HumanMessage(content=query)],
            "query": query, "question_type": "",
            "search_results": [], "answer": "",
        }, config=config)
        elapsed = round(time.time() - start, 2)
        answer = result["messages"][-1].content + self.DISCLAIMER
        return {
            "answer": answer, "thread_id": thread_id,
            "elapsed": elapsed, "messages_count": len(result["messages"]),
        }

    def visualize(self):
        try:
            return self.graph.get_graph().draw_mermaid()
        except Exception as e:
            return f"시각화 실패: {e}"

    def get_history(self, thread_id="default"):
        config = {"configurable": {"thread_id": thread_id}}
        state = self.graph.get_state(config)
        return list(state.values.get("messages", []))

# 테스트
agent = LegalGraphAgent([search_law])
print("=== 그래프 구조 ===")
print(agent.visualize()[:300])

print("\n=== Thread A ===")
print(agent.ask("명예훼손 처벌은?", thread_id="user-A")["answer"][:200])

print("\n=== Thread B ===")
print(agent.ask("사기죄는?", thread_id="user-B")["answer"][:200])

print(f"\nA 이력: {len(agent.get_history('user-A'))}, B 이력: {len(agent.get_history('user-B'))}")

---
## Weekend 2 학습 요약

### 달성한 것들
1. **LegalAgentState** — TypedDict + `Annotated[list, add_messages]`로 상태 정의
2. **노드 분리** — classify / search / analyze / term / general
3. **StateGraph** — 기본 흐름 + 조건부 라우팅 (질문 유형별 분기)
4. **ReAct 패턴** — 도구 호출 ⇄ 에이전트 반복 루프
5. **MemorySaver** — thread_id 별 대화 이력 영속화
6. **Human-in-the-Loop** — 도구 실행 전 interrupt + 사용자 승인
7. **`LegalGraphAgent`** — 통합 그래프 (Weekend 3의 출발점)

### Weekend 3 예고 (Adaptive / Self / Corrective RAG)
- 질문 난이도별 Adaptive Routing (easy / medium / hard)
- Self RAG: 관련성 평가 + 답변 품질 평가 + 재생성 루프
- Corrective RAG: 문서 등급화 + 폴백 검색
- 3종 RAG 통합 → `LegalRAGAgent` 완성